In [3]:
#imports
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
import tldextract
from sklearn.preprocessing import StandardScaler
import joblib

In [4]:
#load raw data
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze("columns")
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze("columns")

In [5]:
#Feature extraction functions
def extract_features(df, url_col='URL'):
    """Extract all 11 features from a DataFrame containing URLs."""
    features = pd.DataFrame(index=df.index)
    urls = df[url_col].astype(str)
    
    # 1. URL length
    features['url_length'] = urls.str.len()
    
    # 2. Number of dots
    features['num_dots'] = urls.str.count(r'\.')
    
    # 3. Number of hyphens
    features['num_hyphens'] = urls.str.count(r'-')
    
    # 4. Number of slashes
    features['num_slashes'] = urls.str.count(r'/')
    
    # 5. Has '@' symbol
    features['has_at'] = urls.str.contains('@').astype(int)
    
    # 6. Has HTTPS
    features['has_https'] = urls.str.startswith('https').astype(int)
    
    # 7. Has IP address (simple regex for IPv4)
    features['has_ip'] = urls.str.contains(r'\d+\.\d+\.\d+\.\d+').astype(int)
    
    # 8. Number of subdomains
    def count_subdomains(url):
        ext = tldextract.extract(url)
        sub = ext.subdomain
        if sub:
            return len(sub.split('.'))
        return 0
    features['num_subdomains'] = urls.apply(count_subdomains)
    
    # 9. Digit ratio
    def digit_ratio(url):
        digits = sum(c.isdigit() for c in url)
        return digits / len(url) if len(url) > 0 else 0
    features['digit_ratio'] = urls.apply(digit_ratio)
    
    # 10. URL entropy (Shannon entropy)
    from collections import Counter
    import math
    def entropy(url):
        prob = [float(url.count(c)) / len(url) for c in set(url)]
        ent = -sum([p * math.log(p) / math.log(2.0) for p in prob])
        return ent
    features['url_entropy'] = urls.apply(entropy)
    
    # 11. Domain length
    def domain_length(url):
        ext = tldextract.extract(url)
        domain = ext.domain
        return len(domain) if domain else 0
    features['domain_length'] = urls.apply(domain_length)
    
    return features

In [6]:
#apply feature extraction
X_train_feat = extract_features(X_train)
X_test_feat = extract_features(X_test)

print("Train features shape:", X_train_feat.shape)
print(X_train_feat.head())

Train features shape: (188296, 11)
   url_length  num_dots  num_hyphens  num_slashes  has_at  has_https  has_ip  \
0          92         6            0           13       0          1       0   
1          30         2            0            2       0          1       0   
2          19         2            0            2       0          0       0   
3          35         2            0            2       0          1       0   
4        5795         5            2            4       1          1       0   

   num_subdomains  digit_ratio  url_entropy  domain_length  
0               2     0.021739     4.371742             11  
1               1     0.000000     3.964735             15  
2               1     0.157895     3.326360              4  
3               1     0.000000     4.050013             19  
4               2     0.131665     4.922899              5  


In [7]:
#Check for any missing/infinite values
print("Missing values:\n", X_train_feat.isnull().sum())
print("Infinite values:\n", np.isinf(X_train_feat).sum())

Missing values:
 url_length        0
num_dots          0
num_hyphens       0
num_slashes       0
has_at            0
has_https         0
has_ip            0
num_subdomains    0
digit_ratio       0
url_entropy       0
domain_length     0
dtype: int64
Infinite values:
 url_length        0
num_dots          0
num_hyphens       0
num_slashes       0
has_at            0
has_https         0
has_ip            0
num_subdomains    0
digit_ratio       0
url_entropy       0
domain_length     0
dtype: int64


In [8]:
#scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_feat)
X_test_scaled = scaler.transform(X_test_feat)

# Convert back to DataFrame (optional, but handy)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_feat.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_feat.columns)

In [9]:
#import joblib
joblib.dump(scaler, '../models/scaler.pkl')
print("Scaler saved.")

Scaler saved.


In [10]:
#save the feature matrices
X_train_scaled.to_csv('../data/processed/X_train_features.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_features.csv', index=False)
y_train.to_csv('../data/processed/y_train_features.csv', index=False)
y_test.to_csv('../data/processed/y_test_features.csv', index=False)
print("Feature matrices saved.")

Feature matrices saved.
